In [ ]:
# preparando o ambiente
!pip install datasets transformers torch

#Imports

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.datasets import fetch_20newsgroups

#Dataset de brinquedo

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando em: {device}")

Rodando em: cuda


In [ ]:
# 2. Carregar Dataset Real do Scikit-Learn (Sem dependência do Hugging Face Hub)
print("Carregando o dataset real '20 Newsgroups' do Scikit-Learn...")
categorias = ['alt.atheism', 'comp.graphics', 'sci.space']

# Buscamos o dataset real filtrado por essas 3 categorias (Portanto, 3 classes: 0, 1 e 2)
dados_reais = fetch_20newsgroups(subset='train', categories=categorias, remove=('headers', 'footers', 'quotes'))

# Selecionamos os primeiros 20 exemplos para o nosso "dataset de brinquedo real"
textos_reais = dados_reais.data
labels_reais = dados_reais.target.tolist()

print(f"Dataset carregado com sucesso!")
print(f"Total de exemplos selecionados: {len(textos_reais)}")
print(f"Classes mapeadas: 0 = Ateísmo, 1 = Computação Gráfica, 2 = Espaço\n")

Carregando o dataset real '20 Newsgroups' do Scikit-Learn...
Dataset carregado com sucesso!
Total de exemplos selecionados: 1657
Classes mapeadas: 0 = Ateísmo, 1 = Computação Gráfica, 2 = Espaço



In [ ]:
class RealToyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Garantir que o texto é uma string limpa
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

#Importando Modelo e Tokenizador

In [ ]:
# 4. Importar o Tokenizador e o BERTimbau para 3 Classes
print("Baixando o BERTimbau do Hub...")
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
model.to(device)

Baixando o BERTimbau do Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

#Instanciando Dataset e DataLoader

In [ ]:
dataset = RealToyDataset(textos_reais, labels_reais, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Loop de Treino

In [ ]:
# 6. Configurar Otimizador Nativo do PyTorch
optimizer = AdamW(model.parameters(), lr=2e-5)

# 7. Loop de Treino de Validação
print("\nIniciando loop de treino com dados reais...")
model.train()

for batch in dataloader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['label'].to(device)

    optimizer.zero_grad()

    # Forward pass
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    logits = outputs.logits

    # Backward pass
    loss.backward()
    optimizer.step()

    predicoes = torch.argmax(logits, dim=1)
    print(f"Batch processado! Loss: {loss.item():.4f} | Predições: {predicoes.tolist()} | Alvos: {labels.tolist()}")

print("\n[SUCESSO] Pipeline multiclasse validado com dados reais do Scikit-Learn e BERTimbau!")


Iniciando loop de treino com dados reais...
Batch processado! Loss: 0.9482 | Predições: [2, 2, 2, 1, 1, 1, 0, 2, 1, 1, 2, 2, 1, 1, 1, 2] | Alvos: [0, 2, 0, 0, 1, 1, 0, 0, 1, 1, 2, 2, 1, 2, 2, 2]
Batch processado! Loss: 1.0250 | Predições: [2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2] | Alvos: [0, 1, 1, 0, 1, 1, 2, 2, 2, 2, 1, 1, 0, 1, 0, 1]
Batch processado! Loss: 0.9743 | Predições: [2, 2, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 2, 1, 2, 2] | Alvos: [2, 2, 1, 2, 0, 0, 0, 2, 1, 1, 1, 1, 1, 0, 2, 0]
Batch processado! Loss: 0.9330 | Predições: [1, 2, 1, 1, 1, 1, 2, 1, 0, 1, 2, 1, 1, 2, 1, 1] | Alvos: [1, 2, 1, 2, 1, 1, 2, 1, 0, 2, 0, 2, 1, 2, 1, 2]
Batch processado! Loss: 0.9613 | Predições: [2, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 1, 1] | Alvos: [2, 2, 2, 0, 2, 1, 2, 2, 2, 1, 0, 0, 1, 0, 2, 0]
Batch processado! Loss: 0.9384 | Predições: [2, 1, 2, 2, 0, 2, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1] | Alvos: [0, 1, 2, 0, 2, 0, 0, 0, 2, 1, 2, 1, 0, 1, 0, 1]
Batch processado! Loss: 0.8076 | Predições: [0, 1